# Project 05: Streamlit Interactive ML Web App

## 1. Project Overview

This project focuses on developing an interactive web application using Streamlit. A previously trained Decision Tree Classification model from Project 04 will be integrated into the application to generate predictions based on user inputs.

The application will use the Titanic dataset to estimate the probability of passenger survival based on passenger class, sex, age, and fare.

## 2. Project Objectives

1. Integrate the trained Machine Learning model.
2. Load the saved model using joblib.
3. Create an interactive Streamlit interface.
4. Accept passenger details through input widgets.
5. Display real-time survival predictions.
6. Handle missing files and invalid inputs.
7. Deploy the application using Streamlit Community Cloud.

In [2]:
from google.colab import files

uploaded = files.upload()

Saving model.joblib to model.joblib


In [3]:
import os

if os.path.exists("model.joblib"):
    print("SUCCESS: Project 04 model uploaded!")
    print("File size:", os.path.getsize("model.joblib"), "bytes")
else:
    print("ERROR: model.joblib was not uploaded.")

SUCCESS: Project 04 model uploaded!
File size: 4073 bytes


In [4]:
import joblib
import pandas as pd
import os

# Load the Project 04 model
loaded_model = joblib.load("model.joblib")

print("MODEL LOADED SUCCESSFULLY")
print("Model type:", type(loaded_model).__name__)

# Verify the expected feature order
FEATURES = ["pclass", "sex", "age", "fare"]

print("\nExpected features:", FEATURES)
print("Model feature count:", loaded_model.n_features_in_)

# Test with one sample passenger
sample = pd.DataFrame(
    [[3, 1, 26.0, 7.925]],
    columns=FEATURES
)

prediction = loaded_model.predict(sample)
probability = loaded_model.predict_proba(sample)

print("\nSample prediction:", prediction[0])
print("Survival probability:", f"{probability[0][1]:.2%}")

print("\nMODEL VERIFICATION COMPLETED")

MODEL LOADED SUCCESSFULLY
Model type: DecisionTreeClassifier

Expected features: ['pclass', 'sex', 'age', 'fare']
Model feature count: 4

Sample prediction: 1
Survival probability: 59.26%

MODEL VERIFICATION COMPLETED


## 3. Streamlit Application Development

The trained Decision Tree Classification model is integrated into a Streamlit web application. Users can enter passenger information through interactive input widgets, including ticket class, sex, age, and fare.

After clicking the Predict button, the application processes the inputs and displays the estimated probability of passenger survival.

In [5]:
%%writefile app.py

import streamlit as st
import pandas as pd
import joblib
import os

# Page configuration
st.set_page_config(
    page_title="Titanic Survival Predictor",
    page_icon="🚢",
    layout="centered"
)

# Application title
st.title("🚢 Titanic Survival Predictor")

st.write(
    "Enter passenger details below to estimate "
    "the probability of survival."
)

# Feature order must match Project 04
FEATURES = ["pclass", "sex", "age", "fare"]

# Load model only once
@st.cache_resource
def load_model():
    return joblib.load("model.joblib")


# Passenger input section
st.subheader("Passenger Information")

pclass = st.selectbox(
    "Ticket Class",
    [1, 2, 3]
)

sex = st.radio(
    "Sex",
    ["Female", "Male"],
    horizontal=True
)

age = st.slider(
    "Age",
    min_value=1,
    max_value=80,
    value=30
)

fare = st.slider(
    "Fare Paid",
    min_value=0.0,
    max_value=500.0,
    value=30.0,
    step=0.5
)

# Prediction button
if st.button("Predict Survival"):

    try:
        # Check whether model file exists
        if not os.path.exists("model.joblib"):
            st.error(
                "Model file is missing. "
                "Please provide model.joblib."
            )
            st.stop()

        # Load trained model
        model = load_model()

        # Encode sex using Project 04 encoding
        sex_encoded = 1 if sex == "Female" else 0

        # Create input DataFrame
        input_data = pd.DataFrame(
            [[pclass, sex_encoded, age, fare]],
            columns=FEATURES
        )

        # Generate prediction
        prediction = model.predict(input_data)[0]

        # Generate survival probability
        probabilities = model.predict_proba(input_data)[0]

        # Find probability corresponding to class 1
        survival_index = list(model.classes_).index(1)

        survival_probability = probabilities[survival_index]

        # Display result
        st.subheader("Prediction Result")

        st.metric(
            "Estimated Chance of Survival",
            f"{survival_probability:.2%}"
        )

        if prediction == 1:
            st.success(
                "The model predicts that the passenger may survive."
            )
        else:
            st.warning(
                "The model predicts that the passenger may not survive."
            )

    except FileNotFoundError:
        st.error(
            "The model file is missing. Please upload model.joblib."
        )

    except Exception as e:
        st.error(
            f"Unable to generate prediction: {e}"
        )

# Educational explanation
st.caption(
    "This educational application uses a Decision Tree model "
    "trained on the historical Titanic dataset. "
    "Predictions are estimates and not guarantees."
)

Writing app.py


In [6]:
import os

if os.path.exists("app.py"):
    print("SUCCESS: app.py created!")
    print("File size:", os.path.getsize("app.py"), "bytes")
else:
    print("ERROR: app.py was not created.")

SUCCESS: app.py created!
File size: 2782 bytes


## 4. Application Dependencies

The application requires Python libraries for building the interactive interface, loading the trained Machine Learning model, and processing passenger input data.

The required libraries are Streamlit, Scikit-learn, Pandas, and Joblib.

In [7]:
%%writefile requirements.txt

streamlit
scikit-learn
pandas
joblib

Writing requirements.txt


In [8]:
import os

if os.path.exists("requirements.txt"):
    print("SUCCESS: requirements.txt created!")

    with open("requirements.txt", "r") as file:
        print("\nFile contents:")
        print(file.read())
else:
    print("ERROR: requirements.txt was not created.")

SUCCESS: requirements.txt created!

File contents:

streamlit
scikit-learn
pandas
joblib



## 5. Project File Verification

The application files are verified before execution. This ensures that the trained model, Streamlit application, and required dependencies are available for the next stage.

In [9]:
import os
import py_compile
import joblib

# Check all required files
required_files = [
    "app.py",
    "model.joblib",
    "requirements.txt"
]

print("========== FILE VERIFICATION ==========")

for filename in required_files:
    if os.path.exists(filename):
        print(f"SUCCESS: {filename} exists")
    else:
        print(f"ERROR: {filename} is missing")

# Check Python syntax
print("\n========== APP SYNTAX CHECK ==========")

try:
    py_compile.compile("app.py", doraise=True)
    print("SUCCESS: app.py has no syntax errors")
except py_compile.PyCompileError as error:
    print("ERROR:", error)

# Check whether model can load
print("\n========== MODEL CHECK ==========")

loaded_model = joblib.load("model.joblib")

print("SUCCESS: Model loaded")
print("Model type:", type(loaded_model).__name__)
print("Feature count:", loaded_model.n_features_in_)

print("\n========== VERIFICATION COMPLETED ==========")

========== FILE VERIFICATION ==========
SUCCESS: app.py exists
SUCCESS: model.joblib exists
SUCCESS: requirements.txt exists

========== APP SYNTAX CHECK ==========
SUCCESS: app.py has no syntax errors

========== MODEL CHECK ==========
SUCCESS: Model loaded
Model type: DecisionTreeClassifier
Feature count: 4

========== VERIFICATION COMPLETED ==========


## 6. Application Dependencies.

This step involves installing the required Python libraries for the Streamlit application. These libraries help load the trained machine learning model, process passenger details, and display survival predictions through an interactive web interface.

In [12]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 114.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 142.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [streamlit]


In [13]:
import sklearn
import pandas
import joblib
import streamlit

print("========== LIBRARY VERSIONS ==========")

print("Scikit-learn:", sklearn.__version__)
print("Pandas:", pandas.__version__)
print("Joblib:", joblib.__version__)
print("Streamlit:", streamlit.__version__)

print("\nVERSION CHECK COMPLETED")

========== LIBRARY VERSIONS ==========
Scikit-learn: 1.6.1
Pandas: 2.2.3
Joblib: 1.6.0
Streamlit: 1.67.0

VERSION CHECK COMPLETED


In [14]:
%%writefile requirements.txt

streamlit==1.67.0
scikit-learn==1.6.1
pandas==2.2.3
joblib==1.6.0

Overwriting requirements.txt


In [15]:
with open("requirements.txt", "r") as file:
    print(file.read())


streamlit==1.67.0
scikit-learn==1.6.1
pandas==2.2.3
joblib==1.6.0



In [16]:
!pip install streamlit
!pip install pyngrok

In [17]:
import os

required_files = ["app.py", "model.joblib", "requirements.txt"]

print("========== PROJECT FILE CHECK ==========")

for file in required_files:
    if os.path.exists(file):
        print(f"✅ {file} - Found")
    else:
        print(f"❌ {file} - Missing")

print("\nFILE CHECK COMPLETED")

========== PROJECT FILE CHECK ==========
✅ app.py - Found
✅ model.joblib - Found
✅ requirements.txt - Found

FILE CHECK COMPLETED


In [18]:
!streamlit run app.py &>/content/streamlit.log &

In [19]:
!cat /content/streamlit.log



2026-11-16 15:37:34.959 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.126.155.190:8501



###Open the Streamlit Application

The Streamlit server is successfully running on port 8501. The application will now be connected to the browser for interactive testing. This allows users to enter passenger details and view Titanic survival predictions.

In [20]:
from google.colab import output

output.serve_kernel_port_as_window(8501)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

The Streamlit server is running successfully on port 8501. The application is now being connected to the Google Colab interface for interactive testing. Users can enter passenger details and view survival predictions.

In [22]:
from google.colab import output

output.serve_kernel_port_as_iframe(8501, height=800)

<IPython.core.display.Javascript object>

### Check Streamlit Server Status

The Streamlit application has been started successfully. However, the application is not displaying correctly in the browser. This step checks the server response and identifies possible loading issues.

In [23]:
import requests

try:
    response = requests.get("http://localhost:8501", timeout=10)

    print("Server Status Code:", response.status_code)

    if response.status_code == 200:
        print("SUCCESS: Streamlit server is responding.")
    else:
        print("The server responded, but needs checking.")

except Exception as e:
    print("Server Connection Error:", e)

Server Status Code: 200
SUCCESS: Streamlit server is responding.


In [24]:
!tail -30 /content/streamlit.log



2026-11-16 15:37:34.959 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.126.155.190:8501

2026-11-16 15:50:52.125 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-ass1c1-2e8mgkh0ameh9-c.asia-southeast1.prod.r.colab.dev, host=m-s-ass1c1-2e8mgkh0ameh9.asia-southeast1-c.c.codatalab-user-runtimes.internal:8007


### Fix the Streamlit Connection

Google Colab uses a browser proxy to display external applications. Sometimes, this proxy blocks WebSocket connections because of origin security restrictions. The Streamlit server settings are adjusted temporarily to allow the application to run inside Colab for testing.

In [25]:
!streamlit run app.py --server.port 8502 --server.address 0.0.0.0 --server.enableCORS false --server.enableXsrfProtection false > /content/streamlit2.log 2>&1 &

In [26]:
!cat /content/streamlit2.log



2026-11-16 15:52:57.585 Uvicorn server started on 0.0.0.0:8502

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8502
  Network URL: http://172.28.0.12:8502
  External URL: http://34.126.155.190:8502



### Display the Application in Colab

The Streamlit application is now running on port 8502 with updated server settings. This step connects the application to the Google Colab interface so that its interactive features can be tested.

In [27]:
from google.colab import output

output.serve_kernel_port_as_iframe(8502, height=800)

<IPython.core.display.Javascript object>

### Prepare Project Files for Deployment

The Streamlit application and machine learning model have been prepared successfully. In this step, all required project files are packaged together for deployment using Streamlit Community Cloud. The deployment will make the application accessible through a public web link.

In [28]:
import zipfile
import os

files = ["app.py", "model.joblib", "requirements.txt"]

with zipfile.ZipFile("Titanic_Survival_Predictor.zip", "w") as zip_file:
    for file in files:
        zip_file.write(file)

print("Project ZIP created successfully!")

print("\nFiles included:")
for file in files:
    print("✅", file)

print("\nZIP file size:", os.path.getsize("Titanic_Survival_Predictor.zip"), "bytes")

Project ZIP created successfully!

Files included:
✅ app.py
✅ model.joblib
✅ requirements.txt

ZIP file size: 7240 bytes


### Download Your Project ZIP

The required application files have been successfully compressed into a ZIP folder. This folder contains the Streamlit application, trained machine learning model, and package requirements. It will be used to create the GitHub repository for deployment.

In [30]:
from google.colab import files

files.download("Titanic_Survival_Predictor.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. Connect GitHub to Streamlit Community Cloud

The Titanic Survival Predictor project files are stored in GitHub. Streamlit Community Cloud will connect to this repository and deploy the machine learning application as a publicly accessible web application.

The Titanic Survival Predictor was tested using two different passenger profiles.

Test Case 1:
- Ticket Class: 3
- Sex: Female
- Age: 26
- Fare: 7.925
- Predicted Survival Probability: 95.93%

Test Case 2:
- Ticket Class: 3
- Sex: Male
- Age: 30
- Fare: 7.925
- Predicted Survival Probability: 13.97%

Both test cases generated prediction results successfully. The application accepted passenger inputs and displayed survival probabilities without errors.

Project 05: Titanic Survival Predictor

Live Application:
https://sriramani-786-titanic-survival-predictor-app-nmir9h.streamlit.app/

GitHub Repository:
https://github.com/SriRamani-786/Titanic-Survival-Predictor

The application is deployed using Streamlit Community Cloud.

## Titanic Survival Predictor

### Project Description
Developed an interactive Machine Learning web application using Streamlit and a Decision Tree Classifier. The application accepts passenger details such as ticket class, sex, age, and fare to estimate the probability of survival.

### Technologies Used
- Python
- Pandas
- Scikit-learn
- Joblib
- Streamlit
- Google Colab
- GitHub

### Live Application Link
https://sriramani-786-titanic-survival-predictor-app-nmir9h.streamlit.app/

## GitHub Repository Link
https://github.com/SriRamani-786/Titanic-Survival-Predictor

### Testing Results
- Test Case 1: Female passenger – 95.93% estimated survival probability.
- Test Case 2: Male passenger – 13.97% estimated survival probability.

The application successfully generated predictions for both test cases.

## 8. Project Conclusion

The Titanic Survival Predictor project successfully demonstrates the deployment of a machine learning model through an interactive Streamlit web application. A Decision Tree classifier was integrated using Joblib to generate survival predictions based on passenger information such as ticket class, sex, age, and fare. The application was tested with different passenger profiles and successfully displayed prediction probabilities. This project provided practical experience in model integration, web application development, GitHub version control, and cloud deployment. The predictions are educational estimates based on historical Titanic data and are not guaranteed outcomes.